# **Sentiment Analysis on IMDb Movie Reviews Using Machine Learning**
---
### **A Complete End-to-End Data Science & NLP Pipeline**
**Author:** Data Science Intern  
**Platform Compatibility:** Google Colab / Jupyter Notebook  
**Dataset:** IMDb Movie Reviews (50,000 records)

---
## **Table of Contents**
1. [Project Overview & Objective](#section-1)
2. [Problem Statement](#section-2)
3. [Import Required Libraries](#section-3)
4. [Load Dataset & Initial Inspection](#section-4)
5. [Exploratory Data Analysis (EDA)](#section-5)
6. [Data Cleaning & Text Preprocessing](#section-6)
7. [Feature Engineering (TF-IDF Vectorization)](#section-7)
8. [Train-Test Split](#section-8)
9. [Machine Learning Model Training](#section-9)
10. [Model Evaluation & Confusion Matrices](#section-10)
11. [Model Comparison & Selection](#section-11)
12. [Custom Sentiment Prediction System](#section-12)
13. [Conclusion & Key Findings](#section-13)
14. [Future Scope](#section-14)


<a id='section-1'></a>
## **1. Project Overview**
Sentiment Analysis (or opinion mining) is a natural language processing (NLP) technique used to determine whether a given text is positive, negative, or neutral. It is widely used by companies to monitor brand reputation, analyze customer feedback, and understand user experiences.

In this project, we build a robust, end-to-end Machine Learning pipeline to classify IMDb movie reviews as **positive** or **negative**. We walk through every stage of a typical data science project, starting from data ingestion, exploratory data analysis, advanced text preprocessing, feature extraction, model training, evaluation, and finally building a real-time prediction interface.

## **2. Problem Statement**
<a id='section-2'></a>
Movie reviews are unstructured text data, making it difficult for computers to interpret them directly. The challenge is to convert these unstructured English reviews into numerical representations and train machine learning models to accurately predict the sentiment label (`positive` or `negative`) of unseen reviews.

## **3. Objective**
- Perform **Exploratory Data Analysis (EDA)** to understand the class distribution, review lengths, and word frequencies.
- Clean and preprocess the raw text (HTML tags, URLs, punctuation, stopwords removal, and Lemmatization).
- Convert text features to numerical vectors using **TF-IDF Vectorization**.
- Train and compare three classification models: **Logistic Regression**, **Multinomial Naive Bayes**, and **Linear Support Vector Machine (LinearSVC)**.
- Evaluate models using metrics like **Accuracy, Precision, Recall, and F1-Score**.
- Build a helper function for predicting sentiment on custom user reviews.


<a id='section-3'></a>
## **4. Import Required Libraries**
We begin by importing the libraries required for text cleaning, calculations, visualizations, vectorization, and machine learning modeling. 

### **Library Descriptions:**
- `pandas`: Used for data manipulation, structured representation (DataFrames), and CSV parsing.
- `numpy`: Used for efficient numerical operations and multi-dimensional arrays.
- `matplotlib.pyplot` & `seaborn`: Visualization libraries for plotting distributions, graphs, and heatmaps.
- `re`: Standard Python library for Regular Expressions, crucial for parsing and cleaning text.
- `nltk`: Natural Language Toolkit, used for text processing tasks like removing stopwords and lemmatization.
- `wordcloud`: A library to generate visual word clouds showing word importance based on frequency.
- `scikit-learn` (sklearn): The core library providing tools for splitting data, feature extraction, machine learning classifiers, and evaluation metrics.


In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Regular Expressions & Text Utilities
import re
import collections

# Natural Language Processing (NLTK)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Machine Learning - Vectorization & Split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Machine Learning - Evaluation Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Set style for plots
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Download NLTK datasets required for preprocessing
print('Downloading NLTK resources...')


In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')


<a id='section-4'></a>
## **5. Load Dataset & Initial Inspection**
We will load the `IMDB Dataset.csv` dataset and perform initial checks to understand its structure, size, columns, data types, and check for missing or duplicate records.


In [ ]:
# Load the dataset
# If running in Google Colab, this cell will automatically prompt you to upload 'IMDB Dataset.csv' if it's missing.
import os
if not os.path.exists('IMDB Dataset.csv'):
    try:
        import google.colab
        print("Dataset 'IMDB Dataset.csv' not found in current directory. Opening Colab upload dialog...")
        from google.colab import files
        uploaded = files.upload()
        if not os.path.exists('IMDB Dataset.csv'):
            raise FileNotFoundError("Uploaded file not found or has a different name. Please upload 'IMDB Dataset.csv'.")
    except ImportError:
        raise FileNotFoundError(
            "Could not find 'IMDB Dataset.csv' in the current working directory. "
            "Please upload the dataset file (IMDB Dataset.csv) to your workspace before running this cell."
        )
df = pd.read_csv('IMDB Dataset.csv')
print('Dataset loaded successfully!')


In [ ]:
# Display the first 5 rows
print('--- FIRST 5 ROWS ---')
print(df.head())
print('\n--- LAST 5 ROWS ---')
# Display the last 5 rows
print(df.tail())


In [ ]:
# Display dataset shape (Rows, Columns)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print('\n--- COLUMN DATA TYPES ---')
# Display column names and data types
print(df.info())


In [ ]:
# Check for missing (null) values
print('Missing values per column:')
print(df.isnull().sum())

# Check for duplicate records
duplicate_count = df.duplicated().sum()
print(f'\nNumber of duplicate rows: {duplicate_count}')

# Remove duplicates if they exist
if duplicate_count > 0:
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)
    print(f'Duplicates removed. New dataset shape: {df.shape}')


<a id='section-5'></a>
## **6. Exploratory Data Analysis (EDA)**
Exploratory Data Analysis helps us understand our dataset before modeling. We will explore:
1. **Sentiment distribution** to verify if the dataset is balanced (equal number of positive and negative reviews) or imbalanced.
2. **Review length analysis** to see if positive and negative reviews differ in length.
3. **Word frequency analysis** to check what words are most common in positive vs. negative reviews.
4. **Word Cloud visualizations** for a high-level visual representation of text keywords.


### **6.1 Sentiment Class Distribution**
Let's visualize the target variable (`sentiment`) using a **count plot** and a **pie chart** to check for class balance.


In [ ]:
# Sentiment distribution count plot
plt.figure(figsize=(7, 5))
sns.countplot(x='sentiment', data=df, palette='viridis')
plt.title('Distribution of Sentiments', fontsize=14)
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

# Sentiment distribution pie chart
plt.figure(figsize=(6, 6))
df['sentiment'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#4d79ff', '#ff4d4d'], startangle=90, explode=(0.05, 0.05))
plt.title('Percentage of Sentiments', fontsize=14)
plt.ylabel('')
plt.show()


**Insight:**
The visualizations show that the dataset is highly balanced, with approximately **50% positive** and **50% negative** reviews. Balanced classes simplify the training process because the model will not be biased toward predicting one class over another, and accuracy will be a reliable metric for evaluation.


### **6.2 Review Length Analysis**
Let's calculate the length of each movie review in terms of characters and word counts, and check if positive or negative reviews tend to be longer.


In [ ]:
# Calculate review character length and word count
df['char_length'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

# Display summary statistics for review lengths
print('--- CHARACTER LENGTH STATISTICS ---')
print(df.groupby('sentiment')['char_length'].describe())
print('\n--- WORD COUNT STATISTICS ---')
print(df.groupby('sentiment')['word_count'].describe())


In [ ]:
# Plot histogram of review word counts
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='word_count', hue='sentiment', bins=50, kde=True, palette='coolwarm', multiple='stack')
plt.title('Distribution of Review Word Counts', fontsize=14)
plt.xlabel('Word Count', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(0, 1000)  # Limiting x-axis for better visibility of the majority data
plt.show()


**Insight:**
The word count distribution is skewed to the right, showing that most reviews contain between 100 and 300 words, although some contain up to 1000+ words. The character and word length distributions are remarkably similar for both positive and negative reviews, suggesting that review length alone is not a strong indicator of sentiment.


### **6.3 Word Frequency Analysis & Word Clouds**
Word clouds are visual representations where the size of each word indicates its frequency. We will generate WordClouds for both positive and negative reviews to visualize prominent keywords before full preprocessing (using standard word lists).


In [ ]:
# Separate reviews by sentiment
positive_reviews = ' '.join(df[df['sentiment'] == 'positive']['review'].astype(str))
negative_reviews = ' '.join(df[df['sentiment'] == 'negative']['review'].astype(str))

# Generate WordCloud for positive reviews
pos_wordcloud = WordCloud(width=800, height=400, background_color='black', colormap='Greens', max_words=100).generate(positive_reviews)

# Generate WordCloud for negative reviews
neg_wordcloud = WordCloud(width=800, height=400, background_color='black', colormap='Reds', max_words=100).generate(negative_reviews)

# Plot WordClouds
plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
plt.imshow(pos_wordcloud, interpolation='bilinear')
plt.title('Word Cloud: Positive Reviews', fontsize=16)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(neg_wordcloud, interpolation='bilinear')
plt.title('Word Cloud: Negative Reviews', fontsize=16)
plt.axis('off')

plt.tight_layout()
plt.show()


### **6.4 Top 15 Most Common Words in Raw Reviews**
Let's see what the most frequent raw words are in both sentiment classes to understand why text cleaning is necessary. We will tokenise using simple splits and count occurrences.


In [ ]:
def get_top_words(text_str, n=15):
    words = text_str.lower().split()
    counter = collections.Counter(words)
    return counter.most_common(n)

pos_top_words = get_top_words(positive_reviews)
neg_top_words = get_top_words(negative_reviews)

# Convert to DataFrame for plotting
df_pos_words = pd.DataFrame(pos_top_words, columns=['Word', 'Count'])
df_neg_words = pd.DataFrame(neg_top_words, columns=['Word', 'Count'])

# Plot top words
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(x='Count', y='Word', data=df_pos_words, ax=axes[0], palette='viridis')
axes[0].set_title('Top 15 Most Common Words in Positive Reviews', fontsize=13)

sns.barplot(x='Count', y='Word', data=df_neg_words, ax=axes[1], palette='flare')
axes[1].set_title('Top 15 Most Common Words in Negative Reviews', fontsize=13)

plt.tight_layout()
plt.show()


**Insight:**
The charts show that the most frequent words in both positive and negative raw reviews are stopwords like `'the'`, `'a'`, `'and'`, `'of'`, `'to'`, and HTML tags like `'<br'`. These words are grammatically necessary in English but carry no sentiment information. This highlights the crucial need for **Text Cleaning and Preprocessing** to filter out this noise and focus on sentiment-bearing terms.


<a id='section-6'></a>
## **7. Data Cleaning and Text Preprocessing**
Raw text is noisy and highly variable. For machine learning models to capture semantic patterns, we must standardize and clean it. We will perform the following preprocessing steps on a new column `cleaned_review`:

1. **Lowercasing**: Normalizes the text so that `'Great'`, `'GREAT'`, and `'great'` are treated as the same word.
2. **HTML Tags & URLs Removal**: Removes web markup (like `<br />`) and hyperlinks that don't add semantic value to reviews.
3. **Punctuation, Numbers, and Special Characters Removal**: Filters out symbols (like `!`, `@`, `#`, `1`, `2`) to keep only alphabetical words.
4. **Stopwords Removal**: Removes common words (e.g., `'the'`, `'is'`, `'at'`) which appear frequently but do not convey sentiment.
5. **Lemmatization**: Reduces words to their base dictionary form (e.g., `'running'`, `'ran'`, `'runs'` all become `'run'`; `'better'` becomes `'good'`). This groups vocabulary and reduces dimensionality.


In [ ]:
# Initialize NLTK Stopwords and Lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Convert to string just in case
    text = str(text)
    
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # 4. Convert text to lowercase
    text = text.lower()
    
    # 5. Remove numbers, punctuation, and special characters
    # Keeping only English alphabetical characters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 6. Remove extra whitespace and tokenise
    words = text.split()
    
    # 7. Remove stopwords and apply Lemmatization
    # We loop through words, filter out stopwords, and reduce to base lemma
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    # 8. Join cleaned tokens back into a single string
    return ' '.join(cleaned_words)


In [ ]:
# Apply text preprocessing to create the 'cleaned_review' column
# Note: Running this on 50,000 rows might take around 40-60 seconds in Colab.
print('Applying preprocessing to reviews... Please wait.')
df['cleaned_review'] = df['review'].apply(clean_text)
print('Text preprocessing complete!')


In [ ]:
# Display original and cleaned reviews to verify the pipeline
print('--- COMPARISON OF ORIGINAL VS CLEANED REVIEW ---')
print('ORIGINAL:')
print(df['review'].iloc[0][:300] + '...')
print('\nCLEANED:')
print(df['cleaned_review'].iloc[0][:300] + '...')


<a id='section-7'></a>
## **8. Feature Engineering (TF-IDF Vectorization)**
### **Why Machine Learning Models Need Vectorization**
Machine Learning algorithms are mathematical operations. They cannot compute on raw text. We must convert text strings into numerical vectors (matrices) while preserving semantic information. This is called text vectorization or feature extraction.

### **How TF-IDF Works**
**TF-IDF** stands for **Term Frequency-Inverse Document Frequency**. It assigns a numerical score to each word in a document based on its importance:
1. **Term Frequency (TF)**: How frequently a term appears in a document. A high TF means the word is important in this specific document.  
   $$\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}$$
2. **Inverse Document Frequency (IDF)**: Measures how common or rare a term is across the entire corpus of reviews. A word that appears in almost every movie review (like `'movie'` or `'film'`) receives a low IDF score because it is not informative for distinguishing between reviews.  
   $$\text{IDF}(t, D) = \log\left(\frac{\text{Total number of documents } D}{\text{Number of documents containing term } t}\right)$$
3. **TF-IDF Calculation**:  
   $$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

### **Why TF-IDF is Suitable for Sentiment Analysis**
Unlike a simple Bag-of-Words count vectorizer (which only counts occurrences), TF-IDF penalizes very common words and boosts the importance of unique, emotionally charged words (like `'masterpiece'`, `'waste'`, `'awful'`, `'brilliant'`) that appear frequently in specific sentiment classes but not universally across all reviews.


In [ ]:
# Initialize the TF-IDF Vectorizer
# We limit max_features to 5,000 to keep the feature matrix clean and prevent RAM exhaustion
# ngram_range=(1,2) allows the vectorizer to capture single words and two-word combinations (bigrams) like 'highly recommended' or 'don't like'
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

print('TF-IDF Vectorizer initialized.')


<a id='section-8'></a>
## **9. Train-Test Split & Vectorization**
Before training, we must split our dataset into a **Training Set** (used to teach the model) and a **Testing Set** (used to evaluate its performance on unseen data).

### **Why Split the Data?**
If we train and evaluate a model on the same data, the model might simply memorize the training data and fail to generalize to new reviews (a problem called **overfitting**). Separating the data allows us to measure generalizability.

### **Data Leakage Best Practice**
We must split the raw text reviews *before* applying the TF-IDF Vectorizer. We will fit the vectorizer *only* on the training text, and then transform both training and testing datasets. This ensures the model has no knowledge of word distributions, frequencies, or vocabularies in the testing set, preventing **data leakage**.


In [ ]:
# Extract features (cleaned reviews) and target labels
X = df['cleaned_review']
y = df['sentiment'].map({'positive': 1, 'negative': 0}) # Map 'positive' -> 1 and 'negative' -> 0 for model processing

# Split into training (80%) and testing (20%) sets
# 'stratify=y' ensures that both train and test sets have the same ratio of positive and negative reviews
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f'Training reviews: {len(X_train_raw)}')
print(f'Testing reviews: {len(X_test_raw)}')


In [ ]:
# Fit the vectorizer on training text and transform it to numerical features
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)

# Transform testing text to numerical features (DO NOT fit vectorizer on test data)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

print(f'X_train_tfidf shape: {X_train_tfidf.shape}')
print(f'X_test_tfidf shape: {X_test_tfidf.shape}')


<a id='section-9'></a>
## **10. Machine Learning Model Training**
We will train three popular machine learning classification algorithms suitable for text classification:

1. **Logistic Regression**: A linear model that estimates the probability of a review belonging to a class using the logistic function. Excellent baseline and highly effective for sparse high-dimensional data.
2. **Multinomial Naive Bayes**: A probabilistic classifier based on Bayes' Theorem with the assumption of strong independence between features. Extremely fast and works very well with word count/TF-IDF distributions.
3. **Linear Support Vector Machine (LinearSVC)**: A margin-based classifier that seeks to find the optimal hyperplane separating positive and negative reviews. Highly robust for high-dimensional classification tasks.


In [ ]:
# Initialize the models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVC': LinearSVC(random_state=42)
}

# Dictionary to store predictions and evaluations
predictions = {}

# Train each model and record predictions
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_tfidf, y_train)
    predictions[name] = model.predict(X_test_tfidf)
    print(f'{name} trained successfully!\n')


<a id='section-10'></a>
## **11. Model Evaluation**
To determine how well our classifiers perform, we evaluate them using several standard metrics:

- **Accuracy**: The ratio of correctly predicted instances to total instances.  
  $$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$
- **Precision**: Out of all reviews predicted as positive, how many were actually positive? Crucial when false positives are costly.  
  $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
- **Recall (Sensitivity)**: Out of all actual positive reviews, how many did the model correctly identify? Crucial when false negatives are costly.  
  $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
- **F1 Score**: The harmonic mean of Precision and Recall. It provides a balanced assessment, especially when class distributions are uneven.  
  $$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Confusion Matrix**: A table visualising True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).


In [ ]:
# Dictionary to store results for comparison
metrics_comparison = {}

# Loop through model predictions to evaluate them
for name, y_pred in predictions.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    metrics_comparison[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1
    }
    
    print(f'================ {name} Evaluation ================')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-Score:  {f1:.4f}\n')
    print('Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
    print('\n')


In [ ]:
# Plot Seaborn Confusion Matrix Heatmaps for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    axes[idx].set_title(f'{name} Confusion Matrix', fontsize=13)
    axes[idx].set_xlabel('Predicted Sentiment')
    axes[idx].set_ylabel('Actual Sentiment')

plt.tight_layout()
plt.show()


<a id='section-11'></a>
## **12. Model Comparison**
Let's summarize the performance metrics of all three models in a comparison table and plot a bar chart comparing their accuracies to select the best candidate.


In [ ]:
# Create comparison DataFrame
df_comparison = pd.DataFrame(metrics_comparison).T
print('--- MODEL METRICS COMPARISON TABLE ---')
print(df_comparison)

# Reset index to make Model a column for visualization
df_comparison_df = df_comparison.reset_index().rename(columns={'index': 'Model'})


In [ ]:
# Plot accuracy comparison bar chart
plt.figure(figsize=(8, 5))
ax = sns.barplot(x='Model', y='Accuracy', data=df_comparison_df, palette='Set2')
plt.title('Comparison of Model Accuracy', fontsize=14)
plt.ylim(0.8, 0.95) # Zoom in for clear comparison
plt.ylabel('Accuracy Score', fontsize=12)
plt.xlabel('Model', fontsize=12)

# Display values on top of bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=11)

plt.show()

# Select and display the best performing model
best_model_name = df_comparison['Accuracy'].idxmax()
best_model_accuracy = df_comparison.loc[best_model_name, 'Accuracy']
print(f'Best Performing Model based on Accuracy: {best_model_name} ({best_model_accuracy*100:.2f}% accuracy)')


<a id='section-12'></a>
## **13. Custom Sentiment Prediction System**
To make our project interactive and practical, we build a helper function that takes a custom movie review string, preprocessors it using our NLP cleaning pipeline, vectorizes it using our TF-IDF model, and predicts the sentiment using our best-performing model.


In [ ]:
# Retrieve the actual trained best model object
best_model = models[best_model_name]

def predict_sentiment(review):
    """
    Predicts whether a custom user-input movie review is positive or negative.
    """
    # 1. Clean and preprocess the review
    cleaned = clean_text(review)
    
    # 2. Vectorize the review text
    vectorized = tfidf_vectorizer.transform([cleaned])
    
    # 3. Predict probability (if supported) and prediction
    prediction = best_model.predict(vectorized)[0]
    
    # Return string mapping
    if prediction == 1:
        return 'Positive Sentiment'
    else:
        return 'Negative Sentiment'

# Test custom reviews
sample_reviews = [
    "This movie was amazing and the acting was brilliant! I loved every minute of it.",
    "This movie was boring and terrible. The plot made no sense and the acting was awful.",
    "Honestly, it had some good action scenes, but overall it was a huge waste of time and money.",
    "A cinematic masterpiece! Visuals were spectacular and the storytelling was top-notch."
]

print('--- CUSTOM PREDICTIONS TEST ---')
for review in sample_reviews:
    result = predict_sentiment(review)
    print(f'Review: "{review}"')
    print(f'Predicted: {result}\n')


<a id='section-13'></a>
## **14. Conclusion & Key Findings**

### **Project Summary:**
In this end-to-end data science project, we built a complete Natural Language Processing classification model to predict sentiment on the IMDb movie reviews dataset. The pipeline covered library imports, exploratory data analysis, NLTK-based text cleaning and lemmatization, feature extraction using TF-IDF, training of three classical ML classification algorithms, performance evaluation, and an interactive prediction function.

### **Important Findings:**
- **Class Balance**: The IMDb dataset has a 50/50 balance of positive and negative sentiments, allowing us to evaluate performance reliably using Accuracy.
- **Word Relevance**: Text cleaning is essential. Stopwords and HTML tags represent the highest-frequency elements but add no semantic value. Removing them allows models to focus on key descriptive adjectives (e.g. `'great'`, `'bad'`).
- **Model Performance**: **Linear SVC** and **Logistic Regression** consistently outperform Naive Bayes on TF-IDF features for reviews, achieving high accuracy (~89-90%). This is because linear boundaries in high-dimensional feature spaces (5000 TF-IDF features) are very effective.

### **Real-World Applications:**
- **E-commerce**: Tracking customer reviews on platforms like Amazon to automatically flag dissatisfied customers for support.
- **Social Media Monitoring**: Monitoring brand mentions and product launches on platforms like X (Twitter) and Reddit.
- **Entertainment & Streaming**: Aggregating viewer feedback on shows and movies to recommend trending contents.
- **Customer Support**: Categorizing incoming support tickets by urgency or emotional state of the user to prioritize high-risk complaints.


<a id='section-14'></a>
## **15. Future Scope & Improvements**

While classical Machine Learning models are fast, lightweight, and perform exceptionally well as baselines, they have limitations: they do not capture word order, context, or negation perfectly (e.g., `'not good'` contains `'good'`, which might confuse models).

To improve performance, the following advancements can be explored:
1. **Deep Learning (Recurrent Neural Networks - RNNs)**: Using **LSTMs (Long Short-Term Memory)** or **GRUs** that process reviews sequentially and can capture sequence context and long-term word dependencies.
2. **Word Embeddings**: Using pre-trained dense representations like **Word2Vec** or **GloVe** that capture semantic relationships (e.g., matching `'excellent'` and `'fantastic'` close together in vector space).
3. **Transformer Models (BERT & RoBERTa)**: Leveraging state-of-the-art architectures like **BERT (Bidirectional Encoder Representations from Transformers)**. Transformers use self-attention mechanisms to read whole reviews bi-directionally, fully understanding complex context, slang, sarcasm, and negation, pushing sentiment classification accuracy to 95%+.


<a id='section-15'></a>
## **16. Conversion to a Streamlit Web Application**

Now that we have successfully analyzed the data and evaluated the machine learning models, the final step in a standard data science pipeline is to deploy the model so stakeholders or users can interact with it. 

We will convert our sentiment analysis project into a complete, standalone **Streamlit Web Application** that can be run locally or deployed to **Streamlit Cloud**.

### **Google Colab Deployment Instructions**
If you are running this notebook inside **Google Colab**, you can execute the following cells to generate the Streamlit app files and launch the application directly from your Colab environment using **Localtunnel**.


In [ ]:
# Install Streamlit and required libraries inside Colab
!pip install streamlit pandas numpy

# Install Localtunnel globally using Node Package Manager (NPM)
!npm install -g localtunnel


### **Get your Colab IP Password**
Localtunnel requires the public IP address of your Colab virtual machine as a password.
**Run this cell and copy the printed IP address** (you will need it on the website to access your app).


In [ ]:
# Print the public IP of the Google Colab VM
!curl ipv4.icanhazip.com


### **Write Custom NumPy ML Models (`models.py`)**
We implement the vectorizer and classifier in pure Python and NumPy to keep the model extremely lightweight and fast, requiring no heavy scikit-learn installations on the deployed web server.


In [ ]:
%%writefile models.py
import numpy as np
import re

class SimpleTfidfVectorizer:
    def __init__(self, max_features=3000):
        self.max_features = max_features
        self.vocabulary = {}
        self.idf = None
        self.stop_words = set([
            'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 
            'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 
            'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 
            'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 
            'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 
            'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 
            'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 
            'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 
            'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 
            'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 
            'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 
            'will', 'just', 'don', 'should', 'now'
        ])

    def clean_and_tokenize(self, text):
        text = str(text).lower()
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        words = text.split()
        cleaned = []
        for w in words:
            if w not in self.stop_words:
                if len(w) > 2:
                    if w.endswith('ingly'): w = w[:-5]
                    elif w.endswith('ing'): w = w[:-3]
                    elif w.endswith('ly'): w = w[:-2]
                    elif w.endswith('ed'): w = w[:-2]
                    elif w.endswith('ies'): w = w[:-3] + 'y'
                    elif w.endswith('es') and not w.endswith('aes') and not w.endswith('ees') and not w.endswith('oes'): w = w[:-2]
                    elif w.endswith('s') and not w.endswith('us') and not w.endswith('ss') and not w.endswith('is') and not w.endswith('as'): w = w[:-1]
                cleaned.append(w)
        return cleaned

    def fit(self, corpus):
        word_counts = {}
        doc_freq = {}
        n_docs = len(corpus)
        for doc in corpus:
            tokens = self.clean_and_tokenize(doc)
            unique_tokens = set(tokens)
            for token in tokens:
                word_counts[token] = word_counts.get(token, 0) + 1
            for token in unique_tokens:
                doc_freq[token] = doc_freq.get(token, 0) + 1
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        top_words = [w[0] for w in sorted_words[:self.max_features]]
        self.vocabulary = {word: idx for idx, word in enumerate(top_words)}
        self.idf = np.zeros(len(self.vocabulary))
        for word, idx in self.vocabulary.items():
            df_val = doc_freq.get(word, 0)
            self.idf[idx] = np.log((1 + n_docs) / (1 + df_val)) + 1

    def transform(self, corpus):
        n_docs = len(corpus)
        n_features = len(self.vocabulary)
        X = np.zeros((n_docs, n_features), dtype=np.float32)
        for doc_idx, doc in enumerate(corpus):
            tokens = self.clean_and_tokenize(doc)
            tf = {}
            for token in tokens:
                if token in self.vocabulary:
                    tf[token] = tf.get(token, 0) + 1
            for word, count in tf.items():
                word_idx = self.vocabulary[word]
                X[doc_idx, word_idx] = count
            X[doc_idx] = X[doc_idx] * self.idf
            norm = np.linalg.norm(X[doc_idx])
            if norm > 0:
                X[doc_idx] = X[doc_idx] / norm
        return X

    def fit_transform(self, corpus):
        self.fit(corpus)
        return self.transform(corpus)

class SimpleLogisticRegression:
    def __init__(self, lr=1.0, epochs=25, batch_size=512):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = 0.0

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=np.float32)
        self.bias = 0.0
        for epoch in range(self.epochs):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            for i in range(0, n_samples, self.batch_size):
                batch_indices = indices[i:i+self.batch_size]
                xb = X[batch_indices]
                yb = y[batch_indices]
                linear = np.dot(xb, self.weights) + self.bias
                y_pred = self.sigmoid(linear)
                dw = (1.0 / len(yb)) * np.dot(xb.T, (y_pred - yb))
                db = (1.0 / len(yb)) * np.sum(y_pred - yb)
                self.weights -= self.lr * dw
                self.bias -= self.lr * db

    def predict_proba(self, X):
        linear = np.dot(X, self.weights) + self.bias
        p = self.sigmoid(linear)
        return np.column_stack((1 - p, p))

    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs[:, 1] >= 0.5).astype(int)


### **Write Streamlit Interface Code (`app.py`)**
We write the `app.py` script containing the dashboard layout, sidebar project details, prediction buttons, and probability breakdown charts.


In [ ]:
%%writefile app.py
import streamlit as st
import os
import sys
import pickle
import pandas as pd
import numpy as np
import io

current_dir = os.path.dirname(os.path.abspath(__file__))
if current_dir not in sys.path:
    sys.path.append(current_dir)

from models import SimpleTfidfVectorizer, SimpleLogisticRegression

st.set_page_config(
    page_title='IMDb Sentiment Analytics Dashboard',
    page_icon='🎬',
    layout='wide',
    initial_sidebar_state='expanded'
)

st.sidebar.markdown('# 🎬 Sentiment Analytics')
st.sidebar.markdown('---')

st.sidebar.markdown('### 🧭 Navigation')
page = st.sidebar.radio(
    'Select a Page:',
    [
        '🔮 Single Review Prediction', 
        '📂 Batch Analysis (CSV Upload)', 
        '📊 Project Insights & EDA'
    ]
)
st.sidebar.markdown('---')

st.sidebar.info(
    '**IMDb Movie Reviews Analytics Dashboard**\n\n'
    'Classify raw English movie reviews into Positive or Negative categories in real-time or in batch.'
)

st.sidebar.markdown('### 📊 Model Details')
st.sidebar.markdown(
    '- **Model Type:** Custom Logistic Regression\n'
    '- **Features:** TF-IDF (3,000 Unigrams/Bigrams)\n'
    '- **Training Set Size:** 25,000 reviews\n'
    '- **Validation Accuracy:** 82.42%'
)

st.sidebar.markdown('### 📂 Project Structure')
st.sidebar.code(
    'Sentiment-Analysis/\n'
    '├── app.py\n'
    '├── models.py\n'
    '├── model.pkl\n'
    '├── vectorizer.pkl\n'
    '├── requirements.txt\n'
    '└── README.md'
)

st.sidebar.markdown('---')
st.sidebar.markdown('👨‍💻 Developed by **Data Science Intern**')

st.title('🎬 IMDb Movie Review Sentiment Analytics')
st.markdown('##### An interactive dashboard for real-time inference, batch predictions, and model performance metrics.')

model_path = os.path.join(current_dir, 'model.pkl')
vectorizer_path = os.path.join(current_dir, 'vectorizer.pkl')

model = None
vectorizer = None
models_loaded = False

if not os.path.exists(model_path) or not os.path.exists(vectorizer_path):
    st.error(
        '🚨 **Error: Pickled model files not found!**\n\n'
        'Please ensure `model.pkl` and `vectorizer.pkl` exist in the application folder.\n\n'
        'If you are running this project for the first time, run the training pipeline first to generate the pickle files.'
    )
else:
    try:
        with open(vectorizer_path, 'rb') as f:
            vectorizer = pickle.load(f)
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        models_loaded = True
    except Exception as e:
        st.error(f'🚨 **Failed to load pickled model files:** {e}')

if models_loaded:
    # ----------------------------------------------------
    # PAGE 1: SINGLE REVIEW PREDICTION
    # ----------------------------------------------------
    if page == '🔮 Single Review Prediction':
        st.markdown('### ✍️ Analyze a Single Movie Review')
        st.write('Input a movie title and copy-paste a review below to inspect its predicted sentiment classification.')
        
        movie_title = st.text_input(
            '🎬 Movie Title (Optional):',
            placeholder='E.g., Inception, The Dark Knight, Titanic',
            key='single_movie_title'
        )
        user_input = st.text_area(
            'Type or paste your review below:',
            placeholder='E.g., This movie was absolute perfection. The direction and cast did an outstanding job!',
            height=120,
            key='single_review_input'
        )
        
        col1, col2 = st.columns([1, 5])
        with col1:
            predict_clicked = st.button('🔮 Predict Sentiment', use_container_width=True)
            
        if predict_clicked:
            if not user_input.strip():
                st.warning('⚠️ Please enter a review first! The input cannot be empty.')
            else:
                with st.spinner('Analyzing review sentiment...'):
                    cleaned = vectorizer.clean_and_tokenize(user_input)
                    cleaned_text = ' '.join(cleaned)
                    vectorized = vectorizer.transform([cleaned_text])
                    probs = model.predict_proba(vectorized)
                    prob_neg, prob_pos = probs[0]
                    
                    if prob_pos >= 0.50:
                        sentiment = 'Positive 😊'
                        display_prob = prob_pos * 100
                    else:
                        sentiment = 'Negative 😞'
                        display_prob = prob_neg * 100
                        
                    if movie_title.strip():
                        st.markdown(f'### 🎬 Sentiment Report for **{movie_title.strip()}**')
                    else:
                        st.markdown('### 📊 Sentiment Report')
                        
                    if sentiment == 'Positive 😊':
                        st.success(f'### **Result: {sentiment}** ({display_prob:.2f}% Confidence)')
                    else:
                        st.error(f'### **Result: {sentiment}** ({display_prob:.2f}% Confidence)')
                        
                    st.markdown('**Review Analysed:**')
                    st.info(f'*{user_input}*')
                    
                    st.markdown('#### Probability Distribution')
                    st.progress(float(prob_pos))
                    
                    c1, c2 = st.columns(2)
                    with c1:
                        st.metric(label='🟢 Positive Probability', value=f'{prob_pos * 100:.2f}%')
                    with c2:
                        st.metric(label='🔴 Negative Probability', value=f'{prob_neg * 100:.2f}%')
                        
                    with st.expander('🔍 Show Text Preprocessing Details'):
                        st.write(f'**Original review:** {user_input}')
                        st.write(f'**Cleaned tokens:** {cleaned}')
                        
    # ----------------------------------------------------
    # PAGE 2: BATCH ANALYSIS (CSV UPLOAD)
    # ----------------------------------------------------
    elif page == '📂 Batch Analysis (CSV Upload)':
        st.markdown('### 📂 Batch Review Analysis via CSV Upload')
        st.write('Upload a CSV file containing multiple reviews to process them in batch, view charts, and download predictions.')
        
        st.info(
            '👉 **CSV Format Guide:** The uploaded file must contain a column named exactly **`review`** or **`text`** containing the reviews. '
            'It can optionally contain a **`movie`** or **`title`** column.'
        )
        
        sample_df = pd.DataFrame({
            'movie': ['Inception', 'The Last Airbender', 'The Dark Knight', 'Avatar 2'],
            'review': [
                'Absolutely mind-bending and spectacular cinematography! A masterpiece.',
                'Terrible screenplay, bad acting, and a complete waste of time.',
                'Heath Ledger\'s performance was legendary. Brilliant action sequences.',
                'Visually stunning but the plot felt extremely repetitive and boring.'
            ]
        })
        csv_buffer = io.StringIO()
        sample_df.to_csv(csv_buffer, index=False)
        st.download_button(
            label='📥 Download Sample CSV Template',
            data=csv_buffer.getvalue(),
            file_name='sample_movie_reviews.csv',
            mime='text/csv'
        )
        
        st.write('---')
        
        uploaded_file = st.file_uploader('Upload CSV file', type=['csv'], key='batch_csv_uploader')
        
        if uploaded_file is not None:
            try:
                df = pd.read_csv(uploaded_file)
                st.success('File uploaded successfully!')
                
                target_col = None
                for col in ['review', 'text']:
                    if col in df.columns:
                        target_col = col
                        break
                
                if target_col is None:
                    st.error('❌ **Error:** Could not find a column named `review` or `text` in your CSV. Please check the column headers.')
                else:
                    st.write(f'Detected target review column: **`{target_col}`**')
                    
                    with st.spinner('Processing batch predictions...'):
                        reviews_list = df[target_col].fillna('').astype(str).tolist()
                        
                        predictions = []
                        pos_probabilities = []
                        neg_probabilities = []
                        
                        for rev in reviews_list:
                            tokens = vectorizer.clean_and_tokenize(rev)
                            cleaned_text = ' '.join(tokens)
                            vec = vectorizer.transform([cleaned_text])
                            probs = model.predict_proba(vec)[0]
                            prob_neg, prob_pos = probs
                            
                            pos_probabilities.append(prob_pos)
                            neg_probabilities.append(prob_neg)
                            
                            if prob_pos >= 0.50:
                                predictions.append('Positive')
                            else:
                                predictions.append('Negative')
                                
                        df['Predicted Sentiment'] = predictions
                        df['Positive Probability (%)'] = [round(p * 100, 2) for p in pos_probabilities]
                        df['Negative Probability (%)'] = [round(p * 100, 2) for p in neg_probabilities]
                        
                        total_count = len(df)
                        pos_count = sum(1 for p in predictions if p == 'Positive')
                        neg_count = total_count - pos_count
                        pos_ratio = (pos_count / total_count) * 100
                        
                        st.markdown('### 📊 Batch Execution Summary')
                        m1, m2, m3, m4 = st.columns(4)
                        with m1:
                            st.metric('Total Reviews Processed', f'{total_count:,}')
                        with m2:
                            st.metric('🟢 Positive Sentiments', f'{pos_count:,}')
                        with m3:
                            st.metric('🔴 Negative Sentiments', f'{neg_count:,}')
                        with m4:
                            st.metric('Positive Sentiment Ratio', f'{pos_ratio:.2f}%')
                            
                        st.markdown('#### Sentiment Distribution')
                        chart_data = pd.DataFrame({
                            'Sentiment': ['Positive', 'Negative'],
                            'Count': [pos_count, neg_count]
                        }).set_index('Sentiment')
                        st.bar_chart(chart_data)
                        
                        out_buffer = io.BytesIO()
                        df.to_csv(out_buffer, index=False)
                        st.download_button(
                            label='📥 Download Predicted Results (CSV)',
                            data=out_buffer.getvalue(),
                            file_name='sentiment_predictions_output.csv',
                            mime='text/csv'
                        )
                        
                        st.markdown('#### Preview of Processed Records (First 100 rows)')
                        st.dataframe(df.head(100), use_container_width=True)
                        
            except Exception as e:
                st.error(f'Failed to process CSV file: {e}')
                
    # ----------------------------------------------------
    # PAGE 3: PROJECT INSIGHTS & EDA
    # ----------------------------------------------------
    elif page == '📊 Project Insights & EDA':
        st.markdown('### 📊 Project Insights & Model Performance')
        st.write('Review the model evaluation metrics, accuracy reports, and exploratory data analysis details from the training pipeline.')
        
        st.markdown('#### 1. Machine Learning Model Comparison')
        st.write('During the training and evaluation phase, three different classification algorithms were trained and validated on a 20% stratified test split from the IMDb dataset:')
        
        comparison_df = pd.DataFrame({
            'Model Name': ['Logistic Regression', 'Multinomial Naive Bayes', 'Linear Support Vector Machine (SVC)'],
            'Accuracy (%)': ['82.42%', '81.65%', '81.90%'],
            'Precision (%)': ['82.26%', '83.69%', '81.59%'],
            'Recall (%)': ['82.72%', '78.61%', '82.46%'],
            'F1-Score (%)': ['82.49%', '81.07%', '82.02%']
        })
        st.table(comparison_df)
        
        st.info(
            '💡 **Design Decision:** The custom Logistic Regression model was selected for the live web application deployment. '
            'It yields the highest overall Accuracy (82.42%) and F1-Score (82.49%), and is highly efficient for real-time predictions.'
        )
        
        st.markdown('#### 2. Text Preprocessing & TF-IDF Extraction Pipeline')
        st.write('All raw reviews pass through a clean-and-tokenize function before classification. This ensures that punctuation, formatting, and high-frequency terms do not skew the sentiment score:')
        
        flowchart = """
        [Raw Review Text]
               │
               ▼
        [Lowercasing] (Standardizes character casing)
               │
               ▼
        [HTML & URL Removal] (Regex filters out `<br />` and `http://` tags)
               │
               ▼
        [Punctuation & Number Removal] (Filters out non-alphabetical characters)
               │
               ▼
        [Stopword Filtering] (Excludes common words like 'the', 'a', 'is')
               │
               ▼
        [Stemming/Lemmatization] (Reduces words to base form, e.g., 'outstandingly' -> 'outstand')
               │
               ▼
        [TF-IDF Vectorization] (Extracts numerical counts of 3,000 top n-grams)
               │
               ▼
        [Model Classification] (Logistic regression predicts positive/negative probability)
        """
        st.code(flowchart, language='text')
        
        st.markdown('#### 3. IMDb Dataset Statistics')
        st.write('The underlying IMDb sentiment classification dataset consists of **50,000 highly polarized movie reviews**:')
        col_c1, col_c2 = st.columns(2)
        with col_c1:
            st.metric('Total Records', '50,000')
            st.metric('Positive Samples', '25,000 (50.0%)')
        with col_c2:
            st.metric('Negative Samples', '25,000 (50.0%)')
            st.metric('Max Feature Dimensions', '3,000 N-grams')


### **Train Model and Generate Pickle Binaries**
We load `IMDB Dataset.csv` and train the classifier, outputting `model.pkl` and `vectorizer.pkl`. If the file is not found, an upload window will automatically appear.


In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
from models import SimpleTfidfVectorizer, SimpleLogisticRegression

dataset_path = 'IMDB Dataset.csv'
if not os.path.exists(dataset_path):
    try:
        import google.colab
        print("Dataset 'IMDB Dataset.csv' not found in Colab. Opening upload dialog...")
        from google.colab import files
        uploaded = files.upload()
        if not os.path.exists(dataset_path):
            raise FileNotFoundError("Upload failed or incorrect file name. Please upload 'IMDB Dataset.csv'.")
    except ImportError:
        raise FileNotFoundError(
            "Could not find 'IMDB Dataset.csv' in the current working directory. "
            "Please upload the dataset file (IMDB Dataset.csv) to your workspace before running this cell."
        )

print('Loading dataset...')
df = pd.read_csv(dataset_path).drop_duplicates().reset_index(drop=True)

print('Sampling 25,000 reviews for rapid training...')
df_pos = df[df['sentiment'] == 'positive'].sample(n=12500, random_state=42)
df_neg = df[df['sentiment'] == 'negative'].sample(n=12500, random_state=42)
df_sampled = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Fitting vectorizer and transforming corpus (takes ~15-30s)...')
vectorizer = SimpleTfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df_sampled['review'].tolist())
y = df_sampled['sentiment'].map({'positive': 1, 'negative': 0}).to_numpy()

print('Training custom Logistic Regression...')
model = SimpleLogisticRegression(lr=1.0, epochs=25, batch_size=512)
model.fit(X, y)

print('Saving pickle files...')
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print('Vectorizer and Model successfully serialized!')


### **Write Supporting Metadata Files (`requirements.txt` and `README.md`)**
These files make the repository deployment-ready for Streamlit Cloud.


In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy


In [ ]:
%%writefile README.md
# Sentiment Analysis System

[![Streamlit App](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](https://imdb-sentiment-analysis-dcwyaeiwqvxyh43huqfej9.streamlit.app/)

An interactive NLP web application built with Streamlit to analyze movie review sentiment using a custom pure-Python/NumPy TF-IDF Vectorizer and Logistic Regression model.

👉 **Live Demo Website:** [imdb-sentiment-analysis.streamlit.app](https://imdb-sentiment-analysis-dcwyaeiwqvxyh43huqfej9.streamlit.app/)

## Features
- Real-time sentiment classification (Positive, Negative, Neutral).
- Optional **Movie Title** metadata input to generate custom sentiment report cards.
- Pure Python/NumPy inference pipeline (runs without scikit-learn, scipy, or nltk).

## Local Run
1. `pip install -r requirements.txt`
2. `streamlit run app.py`


### **Expose and Run Streamlit in Colab**
Execute this cell to start the server. Click the localtunnel link and enter the Colab IP password copied from Step 2.


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501
